In [ ]:
import pandas as pd
import re
import spacy

In [ ]:
file_path="file_path"
df=pd.read_excel(file_path)

In [ ]:


import pandas as pd
import re
import numpy as np
from scipy.stats import mannwhitneyu


TEXT_COL = "English Translation"
SPEAKER_COUNT_COL = "Number of People"
GENDER_COL = "Gender of Speaker(s)"
STANCE_COL = "Stance"


# -----------------------------
# 2) Keep only single-speaker male/female videos
# -----------------------------
df = df[
    (df[SPEAKER_COUNT_COL] == 1) &
    (df[GENDER_COL].isin(["Male", "Female"]))
].copy()



df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

print("Single-speaker sample size:")
print(df[GENDER_COL].value_counts(), "\n")

# -----------------------------
# 3) Feature lexicons

# -----------------------------
CERTAINTY_AMPLIFIERS = [
    "never", "always", "permanent", "permanently", "definitely",
    "certainly", "surely", "absolutely", "undoubtedly", "must"
]

UNIVERSALITY_MARKERS = [
    "all", "every", "everyone", "everybody", "everything",
    "completely", "entire", "entirely", "whole", "totally"
]

FIRST_PERSON_SINGULAR = [
    "i", "me", "my", "mine", "myself"
]

INCLUSIVE_WE_OUR = [
    "we", "our", "ours", "us", "ourselves"
]

HYPERBOLIC_TERMS = [
    "miracle", "miraculous", "amazing", "immortal", "divine",
    "magic", "magical", "incredible", "extraordinary",
    "ultimate", "guaranteed", "instant", "instantly",
    "life-changing", "best", "super"
]

# -----------------------------
# 4) Helpers
# -----------------------------
def normalize_text(text):
    text = text.lower()
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')
    return text

def tokenize(text):
    return re.findall(r"\b[a-z']+\b", normalize_text(text))

def count_terms(text, terms):
    text = normalize_text(text)
    total = 0
    for term in terms:
        # exact word/phrase match
        pattern = r"\b" + re.escape(term.lower()) + r"\b"
        total += len(re.findall(pattern, text))
    return total

def add_feature_counts(frame):
    frame = frame.copy()
    frame["tokens"] = frame[TEXT_COL].apply(lambda x: len(tokenize(x)))

    frame["certainty_count"] = frame[TEXT_COL].apply(lambda x: count_terms(x, CERTAINTY_AMPLIFIERS))
    frame["universality_count"] = frame[TEXT_COL].apply(lambda x: count_terms(x, UNIVERSALITY_MARKERS))
    frame["first_person_I_count"] = frame[TEXT_COL].apply(lambda x: count_terms(x, FIRST_PERSON_SINGULAR))
    frame["inclusive_we_our_count"] = frame[TEXT_COL].apply(lambda x: count_terms(x, INCLUSIVE_WE_OUR))
    frame["hyperbolic_count"] = frame[TEXT_COL].apply(lambda x: count_terms(x, HYPERBOLIC_TERMS))

    # print(frame)

    # normalized versions for fair comparison across transcript lengths
    for col in [
        "certainty_count", "universality_count",
        "first_person_I_count", "inclusive_we_our_count",
        "hyperbolic_count"
    ]:
        frame[col + "_per_1000"] = (frame[col] / frame["tokens"].replace(0, np.nan)) * 1000

    return frame

df = add_feature_counts(df)

# -----------------------------
# 5) Group summaries
# -----------------------------
male = df[df[GENDER_COL] == "Male"].copy()
female = df[df[GENDER_COL] == "Female"].copy()

def safe_sum(x):
    return int(np.nansum(x))

def safe_mean(x):
    return float(np.nanmean(x)) if len(x) else np.nan

def token_pct(count_series, token_series):
    denom = np.nansum(token_series)
    if denom == 0:
        return np.nan
    return (np.nansum(count_series) / denom) * 100

summary_table = pd.DataFrame({
    "Feature": [
        'Certainty amplifiers ("never", "permanent")',
        'Universality markers ("all", "completely")',
        'First-person "I"',
        'Inclusive "we/our"',
        'Hyperbolic terms'
    ],
    "Male speakers": [
        f'{safe_sum(male["certainty_count"])} occurrences',
        f'{token_pct(male["universality_count"], male["tokens"]):.2f}% of tokens',
        f'{safe_mean(male["first_person_I_count"]):.2f} avg per video',
        f'{safe_sum(male["inclusive_we_our_count"])} occurrences',
        f'{safe_sum(male["hyperbolic_count"])} occurrences'
    ],
    "Female speakers": [
        f'{safe_sum(female["certainty_count"])} occurrences',
        f'{token_pct(female["universality_count"], female["tokens"]):.2f}% of tokens',
        f'{safe_mean(female["first_person_I_count"]):.2f} avg per video',
        f'{safe_sum(female["inclusive_we_our_count"])} occurrences',
        f'{safe_sum(female["hyperbolic_count"])} occurrences'
    ]
})


print(summary_table.to_string(index=False))
print()


stats_rows = []
feature_map = {
    "Certainty amplifiers": "certainty_count_per_1000",
    "Universality markers": "universality_count_per_1000",
    'First-person "I"': "first_person_I_count_per_1000",
    'Inclusive "we/our"': "inclusive_we_our_count_per_1000",
    "Hyperbolic terms": "hyperbolic_count_per_1000"
}

for label, col in feature_map.items():
    stats_rows.append({
        "Feature": label,
        "Male mean per 1000 words": round(safe_mean(male[col]), 3),
        "Female mean per 1000 words": round(safe_mean(female[col]), 3),

    })

stats_table = pd.DataFrame(stats_rows)

print(stats_table.to_string(index=False))
print()

